# Solar Irradiance Measurements — Implementation
## 태양복사조도 측정 — 구현

**Paper**: Kopp, G. "Solar Irradiance Measurements", *Living Reviews in Solar Physics* 22:1 (2025).
**DOI**: 10.1007/s41116-025-00040-5

이 노트북은 Kopp (2025) 리뷰의 주요 결과를 합성 (synthetic) 데이터로 재현한다:
1. 3개 태양 주기 (cycles 23-24-25) 위 TSI 시계열
2. 흑점 darkening vs 광반 brightening 성분 분리
3. SSI 파장 의존 변동성 (UV 크고, 가시광 작음)
4. 주기 간 TSI 비교
5. 계측기 degradation 추적 모식도
6. 기후 복사강제 계산 (ΔTSI → ΔT)

This notebook reproduces key results from Kopp (2025) using synthetic data:
1. Multi-cycle TSI time series (cycles 23-24-25)
2. Decomposition into sunspot darkening vs facular brightening
3. SSI variability vs wavelength
4. Cycle-to-cycle TSI comparison
5. Degradation-tracking schematic
6. Climate forcing calculation

**Kernel**: `study-with-ai` conda environment.

In [ ]:
"""Imports and global configuration."""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Physical constants and reference values from Kopp (2025)
TSI_REF = 1361.0  # W/m^2, IAU-accepted nominal TSI at 1 AU
TSI_MIN = 1360.5  # W/m^2, typical solar minimum
TSI_MAX = 1362.0  # W/m^2, typical solar maximum
STEFAN_BOLTZMANN = 5.670374419e-8  # W/m^2/K^4
EARTH_ALBEDO = 0.29
SOLAR_RADIUS_M = 6.957e8  # m
AU_M = 1.495978707e11  # m

print(f"TSI reference (1 AU): {TSI_REF} W/m^2")
print(f"Solar luminosity: {4*np.pi*AU_M**2 * TSI_REF:.3e} W")
print(f"Photospheric flux: {TSI_REF * AU_M**2 / SOLAR_RADIUS_M**2:.3e} W/m^2")
print(f"Effective temperature: {(TSI_REF * AU_M**2 / SOLAR_RADIUS_M**2 / STEFAN_BOLTZMANN)**0.25:.1f} K")

## 1. 합성 TSI 시계열 (3개 주기) / Synthetic TSI Time Series (3 Cycles)

Kopp (2025) Fig. 6에 나타난 PMOD/Community Consensus 합성과 유사한 TSI 변동을 만든다.
- 11년 주기 sinusoidal 변동 ~0.1% peak-to-peak (Cycles 23-24-25)
- 27일 회전 주기 modulation (sunspot passage)
- 단기 노이즈 (convection + p-mode)

We construct TSI variability comparable to the PMOD/Community Consensus composites in Kopp (2025) Fig. 6:
- 11-year sinusoidal ~0.1% peak-to-peak over Cycles 23-24-25
- 27-day rotational modulation (sunspot passage)
- Short-term noise (convection + p-mode).

In [ ]:
def synthetic_tsi_series(start_year: float = 2000.0,
                         end_year: float = 2033.0,
                         cadence_days: float = 1.0,
                         seed: int = 42) -> tuple[np.ndarray, np.ndarray]:
    """Generate a synthetic TSI time series spanning multiple solar cycles.

    Args:
        start_year: Start year (decimal).
        end_year: End year (decimal).
        cadence_days: Sampling cadence in days.
        seed: Random seed for reproducibility.

    Returns:
        Tuple of (time_years, tsi_watts_per_m2).
    """
    rng = np.random.default_rng(seed)
    t = np.arange(start_year, end_year, cadence_days / 365.25)

    # Cycle centers (Cycle 23 max ~2000.5, 24 max ~2014.3, 25 max ~2025.0)
    cycle_centers = np.array([2000.5, 2014.3, 2025.0])
    cycle_amps = np.array([1.25, 0.85, 1.15])  # W/m^2, cycle 24 was weaker
    cycle_width = 5.5  # years (half-width of gaussian envelope)

    envelope = np.zeros_like(t)
    for c, a in zip(cycle_centers, cycle_amps):
        envelope += a * np.exp(-0.5 * ((t - c) / cycle_width) ** 2)

    # 27-day rotational modulation (sunspot passage causes ~-0.1% dips)
    rot_phase = 2 * np.pi * t * 365.25 / 27.0
    rot_amp = 0.3 * (envelope / cycle_amps.max())  # scales with activity
    rotational = -np.abs(rot_amp * np.sin(rot_phase)) * rng.uniform(0.5, 1.5, t.size)

    # Short-term noise: convection + p-modes (~0.01% = 0.14 W/m^2)
    short_term = rng.normal(0.0, 0.05, t.size)

    tsi = TSI_MIN + envelope + rotational + short_term
    return t, tsi


time_yr, tsi = synthetic_tsi_series()
print(f"Range: {time_yr[0]:.1f} to {time_yr[-1]:.1f}, N = {time_yr.size}")
print(f"TSI min: {tsi.min():.2f} W/m^2, max: {tsi.max():.2f} W/m^2")
print(f"Peak-to-peak: {(tsi.max()-tsi.min())/TSI_REF*100:.3f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(time_yr, tsi, color="C3", linewidth=0.4, alpha=0.7)
# Overlay 365-day running mean
window = 365
kernel = np.ones(window) / window
smooth = np.convolve(tsi, kernel, mode="same")
ax.plot(time_yr, smooth, color="k", linewidth=1.5, label="365-day mean")
ax.axhline(TSI_REF, color="gray", linestyle="--", label=f"Reference {TSI_REF} W/m²")
ax.set_xlabel("Year")
ax.set_ylabel("TSI (W m$^{-2}$)")
ax.set_title("Synthetic TSI over Solar Cycles 23-24-25 / 합성 TSI 시계열")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 2. 흑점 Darkening vs 광반 Brightening / Sunspot vs Facular Components

Kopp (2025) Section 4.2-4.3에 따르면, 27일 회전주기 및 11년 주기 TSI 변동은 두 성분의 대립적 결합:
- **Sunspot darkening**: 흑점이 disk를 가로지를 때 TSI -0.1% ~ -0.34% (short-term, days)
- **Facular brightening**: 광반은 UV에서 특히 밝음 (weeks-months, longer-lived)
- **Net effect**: 시간 평균시 광반이 우세 → TSI는 활동 최대일 때 **높다**

Per Kopp (2025) Sec. 4.2-4.3: TSI variability on rotation and cycle timescales is the opposing combination of sunspot darkening (short, ~0.1-0.34% dips) and facular brightening (longer, net positive). Time-averaged, faculae dominate, making TSI higher at cycle maximum.

In [ ]:
def sunspot_facular_decomposition(time_yr: np.ndarray,
                                  seed: int = 7) -> tuple[np.ndarray, np.ndarray]:
    """Decompose TSI variability into sunspot (negative) and facular (positive) components.

    Follows the SATIRE-like conceptual model: each cycle has intermittent sunspot
    dips and a broader facular envelope. The net TSI is their sum above quiet Sun.

    Args:
        time_yr: Decimal-year time array.
        seed: RNG seed.

    Returns:
        Tuple of (sunspot_component, facular_component) in W/m^2.
    """
    rng = np.random.default_rng(seed)
    cycle_centers = np.array([2000.5, 2014.3, 2025.0])
    cycle_width = 5.5
    activity = np.zeros_like(time_yr)
    for c in cycle_centers:
        activity += np.exp(-0.5 * ((time_yr - c) / cycle_width) ** 2)
    activity /= activity.max()  # 0-1

    # Facular component: smooth positive envelope, peak +2 W/m^2 at maximum
    facular = 2.2 * activity

    # Sunspot component: intermittent negative dips (probability scales with activity)
    sunspot = np.zeros_like(time_yr)
    n = time_yr.size
    for i in range(n):
        if rng.random() < 0.015 * activity[i]:  # dip occurs
            # Dip duration ~5-10 days (here in array indices)
            dip_len = rng.integers(5, 15)
            dip_depth = rng.uniform(0.5, 3.0) * activity[i]
            end = min(i + dip_len, n)
            sunspot[i:end] -= dip_depth
    return sunspot, facular


spot, facula = sunspot_facular_decomposition(time_yr)
net = spot + facula

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(time_yr, facula, color="orange", label="Facular brightening / 광반", lw=1.2)
ax[0].plot(time_yr, spot, color="navy", label="Sunspot darkening / 흑점", lw=0.5, alpha=0.7)
ax[0].set_ylabel("Component (W m$^{-2}$)")
ax[0].legend(loc="upper right")
ax[0].set_title("Decomposition: Faculae vs Sunspots / 광반-흑점 분해")

ax[1].plot(time_yr, net, color="C3", lw=0.7)
ax[1].axhline(0, color="k", lw=0.5)
ax[1].set_ylabel("Net ΔTSI (W m$^{-2}$)")
ax[1].set_xlabel("Year")
ax[1].set_title("Net TSI variation (facular - sunspot) / 순 TSI 변동")
plt.tight_layout()
plt.show()

print(f"Time-averaged net TSI offset: {net.mean():.3f} W/m² (positive = facular-dominated)")

## 3. SSI 파장별 변동성 / SSI Variability vs Wavelength

Kopp et al. (2024) 및 본 리뷰 Eq.:
$$\frac{\Delta SSI}{SSI}\approx \frac{5}{8\lambda}\cdot\frac{\Delta TSI}{TSI}\quad (\lambda\ \text{in}\ \mu m)$$
UV (<300 nm)에서는 깨지지만 visible/NIR에서는 유효. Lyα 같은 UV 라인은 factor 2 (100%) 변동.

The Kopp et al. (2024) approximation holds in the visible/NIR. UV lines (e.g., Lyα at 121.6 nm) vary by factors up to 2 (100%).

In [ ]:
def ssi_cycle_variability(wavelength_nm: np.ndarray,
                          delta_tsi_frac: float = 1e-3) -> np.ndarray:
    """Estimate cycle-amplitude SSI variability vs wavelength.

    Uses Kopp et al. (2024) approximation for visible/NIR and empirical
    enhancement factors for UV based on Rottman (2005) and Snow et al. (2022).

    Args:
        wavelength_nm: Wavelengths in nm.
        delta_tsi_frac: Cycle-amplitude fractional TSI variation (default 0.1%).

    Returns:
        Fractional SSI variability (dimensionless).
    """
    lam_um = wavelength_nm / 1000.0
    kopp_approx = (5.0 / (8.0 * lam_um)) * delta_tsi_frac

    # UV enhancement: factor 2 at 121 nm (Lyα), blending smoothly above 300 nm
    uv_enhance = np.ones_like(lam_um)
    mask_uv = wavelength_nm < 300
    # Empirical: at 121 nm -> 2.0 (100%), at 200 nm -> 0.10, at 300 nm -> 0.03
    uv_table_nm = np.array([120, 150, 180, 200, 250, 300])
    uv_table_frac = np.array([1.0, 0.30, 0.15, 0.10, 0.05, 0.03])
    uv_interp = np.interp(wavelength_nm[mask_uv], uv_table_nm, uv_table_frac)
    out = np.where(mask_uv, uv_interp, kopp_approx)
    return out


wl = np.logspace(np.log10(100), np.log10(2400), 300)
frac_var = ssi_cycle_variability(wl)

fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(wl, frac_var * 100, color="C2", lw=1.5)
ax.axhline(0.1, color="k", linestyle="--", alpha=0.5, label="TSI (0.1%)")
ax.axvline(121.6, color="red", linestyle=":", alpha=0.5, label="Lyα")
ax.axvspan(100, 300, alpha=0.1, color="violet", label="UV")
ax.axvspan(400, 700, alpha=0.1, color="yellow", label="Visible")
ax.axvspan(700, 2400, alpha=0.1, color="orangered", label="NIR")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("ΔSSI/SSI over solar cycle (%)")
ax.set_title("SSI cycle-amplitude variability vs wavelength / 파장별 SSI 변동성")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

for w in [121.6, 200, 400, 650, 1000, 2000]:
    v = np.interp(w, wl, frac_var) * 100
    print(f"λ={w:>7.1f} nm : ΔSSI/SSI ≈ {v:6.3f}%")

## 4. 주기간 TSI 비교 / Cycle-to-Cycle TSI Comparison

PMOD composite (Kopp 2025 Sec 4.3)에 따르면 각 주기의 peak-to-peak 변동:
- Cycle 21 (1980 peak): 0.082%
- Cycle 22 (1989 peak): 0.077%
- Cycle 23 (2000 peak): 0.096%
- Cycle 24 (2014 peak): 0.063% (weaker cycle)

PMOD composite cycle-amplitude variability (Kopp 2025 Sec 4.3). Cycle 24 was notably weaker.

In [ ]:
def cycle_slice(t: np.ndarray, tsi: np.ndarray,
                center_year: float, halfwidth: float = 5.5) -> tuple[np.ndarray, np.ndarray]:
    """Extract a solar-cycle slice centered at ``center_year`` ± ``halfwidth`` years.

    Args:
        t: Decimal-year time array.
        tsi: TSI array matching ``t``.
        center_year: Cycle-maximum year.
        halfwidth: Half-cycle width in years.

    Returns:
        Tuple of (relative_time, tsi_slice).
    """
    mask = np.abs(t - center_year) < halfwidth
    return t[mask] - center_year, tsi[mask]


# Compare Cycles 23/24/25 (from our synthetic)
fig, ax = plt.subplots(figsize=(10, 5))
colors = {"Cycle 23": "C0", "Cycle 24": "C1", "Cycle 25": "C3"}
centers = {"Cycle 23": 2000.5, "Cycle 24": 2014.3, "Cycle 25": 2025.0}

for lbl, c in centers.items():
    rt, ts = cycle_slice(time_yr, tsi, c)
    # Smooth for presentation
    if rt.size > 365:
        kernel = np.ones(365) / 365
        ts_smooth = np.convolve(ts, kernel, mode="same")
    else:
        ts_smooth = ts
    peak = ts_smooth.max()
    trough = ts_smooth.min()
    amp_pct = (peak - trough) / TSI_REF * 100
    ax.plot(rt, ts_smooth, color=colors[lbl],
            label=f"{lbl}: Δ={amp_pct:.3f}%", lw=1.5)

ax.set_xlabel("Years from cycle maximum")
ax.set_ylabel("TSI (W m$^{-2}$)")
ax.set_title("Cycle-to-cycle TSI comparison / 태양주기별 TSI 비교")
ax.legend()
plt.tight_layout()
plt.show()

## 5. 계측기 Degradation 추적 / Instrument Degradation Tracking

Kopp (2025) Section 2.3.4에 따르면, 중복 채널 (primary + rarely-used secondary)을 duty-cycle 비율로 운용해 UV-induced black-paint degradation을 추적하고 보정한다. 예: VIRGO PMO6 (150:1), SORCE/TIM (4-way).

Per Kopp (2025) Sec. 2.3.4, duty-cycled redundant channels (primary 150× or 1200× more exposed than secondary) track UV-induced black-paint degradation. Post-flight data processing corrects the primary using the secondary as a stable reference.

In [ ]:
def degradation_model(time_yr: np.ndarray,
                      exposure_fraction: float,
                      tau_years: float = 5.0,
                      total_decline: float = 0.005) -> np.ndarray:
    """Model exponential decline in detector sensitivity from cumulative exposure.

    Args:
        time_yr: Decimal-year time array.
        exposure_fraction: Fraction of total mission time the channel is exposed (0-1).
        tau_years: Characteristic decay timescale for 1/e loss at full exposure.
        total_decline: Asymptotic fractional sensitivity loss at full exposure.

    Returns:
        Sensitivity fraction (1 = perfect, < 1 after degradation).
    """
    t0 = time_yr[0]
    cumulative_exposure = (time_yr - t0) * exposure_fraction
    return 1.0 - total_decline * (1.0 - np.exp(-cumulative_exposure / tau_years))


# Primary: 99% duty cycle ; Secondary: 1% duty cycle (VIRGO PMO6-style)
t_deg = np.linspace(2017.0, 2032.0, 2000)
primary = degradation_model(t_deg, exposure_fraction=0.99)
secondary = degradation_model(t_deg, exposure_fraction=0.01)
correction = secondary / primary  # multiplicative correction to apply to primary

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(t_deg, primary * 100, label="Primary (99% duty)", color="C3", lw=1.5)
ax[0].plot(t_deg, secondary * 100, label="Secondary (1% duty)", color="navy", lw=1.5)
ax[0].set_xlabel("Year")
ax[0].set_ylabel("Sensitivity (%)")
ax[0].set_title("On-orbit sensitivity / 궤도상 민감도")
ax[0].legend()

ax[1].plot(t_deg, (correction - 1) * 1e6, color="C2", lw=1.5)
ax[1].set_xlabel("Year")
ax[1].set_ylabel("Correction factor - 1 (ppm)")
ax[1].set_title("Derived degradation correction / 도출된 보정값")
plt.tight_layout()
plt.show()

print(f"Primary loss after 15 yr: {(1 - primary[-1]) * 100:.3f}%")
print(f"Secondary loss after 15 yr: {(1 - secondary[-1]) * 100:.4f}%")
print(f"TSIS-1/TIM actual degradation rate: 0.00025 %/yr (best ever)")

## 6. 기후 복사강제 계산 / Climate Radiative Forcing

Kopp (2025) Section 1.1에 따른 간단한 에너지 수지:
$$T_E = \left[\frac{(1-A)S_0}{4\varepsilon\sigma}\right]^{1/4}$$
$\Delta T/\Delta S = (1-A)/(16\varepsilon\sigma T^3)$, 기후 감도 $\sim 0.1$ K per W/m^2 (Lean & Matthes 2017).

Simple energy-balance from Kopp (2025) Sec 1.1; climate sensitivity ~0.1 K per W/m^2 (Lean & Matthes 2017).

In [ ]:
def equilibrium_temperature(tsi: float, albedo: float = EARTH_ALBEDO,
                            emissivity: float = 1.0) -> float:
    """Compute gray-body equilibrium temperature from TSI.

    Args:
        tsi: Total solar irradiance at 1 AU in W/m^2.
        albedo: Earth's Bond albedo.
        emissivity: Effective thermal emissivity (1 for blackbody).

    Returns:
        Equilibrium temperature in K.
    """
    return ((1 - albedo) * tsi / (4 * emissivity * STEFAN_BOLTZMANN)) ** 0.25


def radiative_forcing(delta_tsi: float, albedo: float = EARTH_ALBEDO) -> float:
    """Compute top-of-atmosphere radiative forcing from TSI change.

    Args:
        delta_tsi: Change in TSI (W/m^2).
        albedo: Earth's Bond albedo.

    Returns:
        Radiative forcing ΔF at TOA (W/m^2).
    """
    return (1 - albedo) / 4 * delta_tsi


# Scenarios from Kopp (2025)
scenarios = {
    "TSI = 1361 (reference)": 0.0,
    "11-yr cycle ΔTSI = +1.36 W/m² (+0.1%)": 1.36,
    "Maunder Minimum ΔTSI = -1.36 W/m²": -1.36,
    "Large excursion ΔTSI = -4.0 W/m²": -4.0,
}
climate_sensitivity_K_per_Wm2 = 0.5  # transient, IPCC-range lower bound

print(f"Reference equilibrium T (α=0.29, ε=1): {equilibrium_temperature(TSI_REF):.2f} K")
print("\nScenario analysis / 시나리오 분석:")
print(f"{'Scenario':<42} {'ΔTSI':>8} {'ΔF':>8} {'ΔT':>6}")
print("-" * 68)
for name, dtsi in scenarios.items():
    df = radiative_forcing(dtsi)
    dt = df * climate_sensitivity_K_per_Wm2
    print(f"{name:<42} {dtsi:+7.2f}  {df:+7.3f}  {dt:+5.3f}")

In [ ]:
# Plot radiative forcing vs TSI change, with climate sensitivity band
dtsi_range = np.linspace(-5, 5, 200)
df_range = radiative_forcing(dtsi_range)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(dtsi_range, df_range, color="C0", lw=1.5)
ax[0].axvline(1.36, color="orange", linestyle="--", label="11-yr cycle peak")
ax[0].axvline(-1.36, color="purple", linestyle="--", label="Maunder Min estimate")
ax[0].set_xlabel("ΔTSI (W m$^{-2}$)")
ax[0].set_ylabel("ΔF at TOA (W m$^{-2}$)")
ax[0].set_title("Radiative forcing vs ΔTSI / 복사강제력")
ax[0].legend()

sens_range = np.array([0.3, 0.5, 1.0])  # K per W/m^2
for s in sens_range:
    ax[1].plot(dtsi_range, df_range * s, label=f"λ={s} K/(W/m²)")
ax[1].set_xlabel("ΔTSI (W m$^{-2}$)")
ax[1].set_ylabel("ΔT (K)")
ax[1].set_title("Temperature response / 온도 반응")
ax[1].legend()
ax[1].axhline(0, color="k", lw=0.5)
plt.tight_layout()
plt.show()

## 7. 요약 / Summary

본 노트북은 Kopp (2025)의 핵심 관측적 결과를 합성 데이터로 재현했다:

| 결과 / Result | 값 / Value |
|---|---|
| 기준 TSI / Reference TSI | 1361 W/m² (IAU 2015) |
| 태양 유효온도 / T_eff | 5772 K |
| 지구 회색체 평형 온도 / Earth gray-body T | ~255 K |
| 11-yr TSI 변동 / Cycle amplitude | ~0.1% peak-to-peak |
| 11-yr ΔT (기후 반응) / Climate response | ~0.1 K |
| UV (Lyα) 변동 / Lyα variability | ~100% (factor 2) |
| TSIS-1/TIM 안정도 / TSIS-1/TIM stability | 0.00025 %/yr (best ever) |
| Measurement requirement (stability) | <0.001 %/yr |

This notebook reproduced the key observational results of Kopp (2025) using synthetic data. The primary scientific insights are:
- TSI varies in-phase with the 11-year solar cycle, with faculae dominating over sunspots on time averages;
- UV variability vastly exceeds TSI variability, driving stratospheric climate coupling;
- Instrument degradation tracking via duty-cycled redundancy is essential for multi-decadal records;
- The 47-year record still lacks the stability to definitively detect secular solar trends.

### Connections / 연결
- **Paper #11 Haigh (2007)**: 태양-기후 커플링의 물리 → 본 논문의 측정 입력
- **IPCC AR6**: Community Consensus TSI Composite 사용
- **Future (TSIS-2, TRUTHS, CLARREO)**: <0.001%/yr 안정도 달성 목표